In [1]:
import os
import json

# =====================================================
# CONFIGURAÇÕES
# =====================================================

INPUT_PATH = "../database/"
OUTPUT_PATH = "benchmark_prepared/"

# True = manter apenas séries com comprimento máximo
# False = manter todas as séries
FILTER_MAX_LENGTH = True

os.makedirs(OUTPUT_PATH, exist_ok=True)

DATASETS = {
    # "m3_monthly_dataset.tsf": {
    #     "name": "m3_monthly",
    #     "horizon": 18
    # },
    # "m4_monthly_dataset.tsf": {
    #     "name": "m4_monthly",
    #     "horizon": 18
    # },
    # "nn5_weekly_dataset.tsf": {
    #     "name": "nn5_weekly",
    #     "horizon": 8
    # },
    # "tourism_monthly_dataset.tsf": {
    #     "name": "tourism_monthly",
    #     "horizon": 24
    # },
    # "cif_2016_dataset.tsf": {
    #     "name": "cif_2016",
    #     "horizon": 12
    # }
    "hospital_dataset.tsf": {
        "name": "hospital",
        "horizon": 12
    }
}


# =====================================================
# PARSE DO ARQUIVO TSF
# =====================================================

def parse_tsf(file_path):

    sequences = []
    horizons = []

    with open(file_path, "r", encoding="latin1") as f:

        data_started = False

        for line in f:

            line = line.strip()

            if not line:
                continue

            if line.startswith("@data"):
                data_started = True
                continue

            if not data_started:
                continue

            parts = line.split(":")

            if len(parts) == 4:
                _, _, horizon, values = parts
                horizon = int(horizon)

            elif len(parts) == 3:
                _, _, values = parts
                horizon = None

            else:
                continue

            series = []

            for v in values.split(","):

                v = v.strip()

                if v == "?" or v == "":
                    continue

                try:
                    series.append(float(v))
                except:
                    continue

            if len(series) > 0:
                sequences.append(series)
                horizons.append(horizon)

    return sequences, horizons


# =====================================================
# FILTRAR SERIES COM TAMANHO MAXIMO
# =====================================================

def filter_max_length(sequences):

    if len(sequences) == 0:
        return sequences, 0

    max_len = max(len(s) for s in sequences)

    filtered = [s for s in sequences if len(s) == max_len]

    return filtered, max_len


# =====================================================
# OBTER COMPRIMENTO MAXIMO
# =====================================================

def get_max_length(sequences):

    if len(sequences) == 0:
        return 0

    return max(len(s) for s in sequences)


# =====================================================
# SALVAR JSONL
# =====================================================

def save_jsonl(sequences, output_file):

    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    with open(output_file, "w", encoding="utf-8") as f:

        for seq in sequences:

            json_line = {"sequence": seq}

            f.write(json.dumps(json_line) + "\n")


# =====================================================
# PROCESSAMENTO PRINCIPAL
# =====================================================

for file_name, config in DATASETS.items():

    print("\nProcessing:", file_name)

    dataset_path = os.path.join(INPUT_PATH, file_name)

    sequences, horizons = parse_tsf(dataset_path)

    dataset_name = config["name"]
    horizon = config["horizon"]

    total_series_original = len(sequences)

    # =================================================
    # CIF 2016
    # =================================================
    # O arquivo TSF não possui horizonte por linha.
    # Então simplesmente usamos todas as séries
    # e definimos horizon=12.

    if dataset_name == "cif_2016":

        filtered_sequences = []

        for seq, h in zip(sequences, horizons):

            if h is None or h == 12:
                filtered_sequences.append(seq)

        sequences = filtered_sequences

    total_series_after_horizon_filter = len(sequences)

    if len(sequences) == 0:
        print("WARNING: No series found after filtering")
        continue

    # =================================================
    # FILTRAR POR COMPRIMENTO MAXIMO (OPCIONAL)
    # =================================================

    max_len = get_max_length(sequences)

    if FILTER_MAX_LENGTH:

        sequences, max_len = filter_max_length(sequences)

    total_series_final = len(sequences)

    # =================================================
    # SALVAR DATASET
    # =================================================

    output_file = os.path.join(
        OUTPUT_PATH,
        dataset_name,
        f"horizon_{horizon}",
        "dataset.jsonl"
    )

    save_jsonl(sequences, output_file)

    print(
        f"Dataset: {dataset_name} | "
        f"original_series={total_series_original} | "
        f"after_horizon_filter={total_series_after_horizon_filter} | "
        f"final_series={total_series_final} | "
        f"max_length={max_len} | "
        f"horizon={horizon} | "
        f"filtered_max_length={FILTER_MAX_LENGTH}"
    )


print("\nAll datasets processed successfully.")


Processing: hospital_dataset.tsf
Dataset: hospital | original_series=767 | after_horizon_filter=767 | final_series=767 | max_length=84 | horizon=12 | filtered_max_length=True

All datasets processed successfully.
